# 03 · Text Classification (Fiction / Nonfiction)

Maps the raw `categories` column into a small set of buckets, then uses zero-shot classification (`facebook/bart-large-mnli`) to fill in books whose category didn't map to any bucket.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import CLEANED_BOOKS_CSV, FICTION_CATEGORIES
from src.features.classification import (
    map_simple_categories, load_zero_shot_pipeline, predict_label,
    evaluate_classifier, classify_missing_categories, build_categorized_books,
)

## 1. Load cleaned books & map simple categories

In [ ]:
books = pd.read_csv(CLEANED_BOOKS_CSV)
books = map_simple_categories(books)
books[~(books["simple_categories"].isna())]

## 2. Load the zero-shot classifier

Set `device=0` below if you have a CUDA GPU available.

In [ ]:
pipe = load_zero_shot_pipeline(device=-1)

## 3. Sanity-check it against books with a known label

In [ ]:
accuracy = evaluate_classifier(books, pipe, n_samples_per_class=300)
accuracy

## 4. Fill in the missing categories

In [ ]:
books = classify_missing_categories(books, pipe, candidate_labels=FICTION_CATEGORIES)

## 5. Save

(equivalent to calling `build_categorized_books()` directly — shown step by step above for exploration purposes)

In [ ]:
from src.config import BOOKS_WITH_CATEGORIES_CSV
books.to_csv(BOOKS_WITH_CATEGORIES_CSV, index=False)
print("Saved:", books.shape, "->", BOOKS_WITH_CATEGORIES_CSV)

---
Next notebook: **04_sentiment_analysis.ipynb**